# Fine-tune XLM-R for dzhynsa (binary: ad / not ad)

Kaggle settings: **Accelerator = GPU T4**, **Internet = On**.
Input: the private dataset `dzhynsa-text` made by `python -m tse_dzhynsa_bot.export_text`.

Output (`/kaggle/working/xlmr_dzhynsa.zip`): `model.onnx` (int8, for CPU serving),
`tokenizer.json`, `preds_val.parquet`, `preds_test.parquet`, `train_meta.json`.
Download the zip and run `python -m tse_dzhynsa_bot.import_xlmr --zip <path>` on the laptop.
Takes about 25-35 min on one T4.

In [ ]:
# !pip install -q onnx==1.19.1 onnxruntime==1.23.2
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "onnx==1.19.1", "onnxruntime==1.23.2"],
    check=True,
)

In [ ]:
import copy
import json
import os
import random
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
from sklearn.metrics import average_precision_score
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)


def find_data_dir() -> Path:
    if os.environ.get("DATA_DIR"):
        return Path(os.environ["DATA_DIR"])
    hits = sorted(Path("/kaggle/input").rglob("manifest.json"))
    assert hits, "attach the dzhynsa-text dataset (Add Input)"
    return hits[0].parent


CFG = {
    "model_name": os.environ.get("MODEL_NAME", "FacebookAI/xlm-roberta-base"),
    "max_len": 256,
    "head_tokens": 128,
    "epochs": int(os.environ.get("EPOCHS", 3)),
    "lr": 2e-5,
    "batch_size": 32,
    "warmup": 0.06,
    "weight_decay": 0.01,
    "pos_weight": 10.0,  # ~0.7% of posts are ads: weight the ad class in the loss
    "seed": 0,
}
DATA_DIR = find_data_dir()
OUT_DIR = Path(os.environ.get("OUT_DIR", "/kaggle/working/xlmr_dzhynsa"))
TMP_DIR = Path(os.environ.get("TMP_DIR", "/tmp/xlmr"))  # big files that must not go to output
MAX_ROWS = int(os.environ.get("MAX_ROWS", 0))  # smoke tests only; 0 = all rows
OUT_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"
print(DATA_DIR, device, torch.__version__, transformers.__version__)

In [ ]:
manifest = json.loads((DATA_DIR / "manifest.json").read_text())
parts = {k: pd.read_parquet(DATA_DIR / f"{k}.parquet") for k in ("train", "val", "test")}
if MAX_ROWS:
    parts = {k: v.head(MAX_ROWS) for k, v in parts.items()}
for k, v in parts.items():
    print(k, len(v), "ads:", int(v["label_ad"].sum()))

tok = AutoTokenizer.from_pretrained(CFG["model_name"])
backend = tok.backend_tokenizer
backend.no_truncation()
backend.no_padding()

In [ ]:
# Same as encode() in src/tse_dzhynsa_bot/xlmr_model.py: serving cuts texts the same way.
def encode(tokenizer, texts, max_len=CFG["max_len"], head=CFG["head_tokens"]):
    cls, sep, pad = (tokenizer.token_to_id(t) for t in ("<s>", "</s>", "<pad>"))
    body = max_len - 2
    rows = []
    for enc in tokenizer.encode_batch([t or "" for t in texts], add_special_tokens=False):
        ids = enc.ids
        if len(ids) > body:
            ids = ids[:head] + ids[-(body - head) :]
        rows.append([cls, *ids, sep])
    width = max(len(r) for r in rows)
    input_ids = np.full((len(rows), width), pad, dtype=np.int64)
    mask = np.zeros((len(rows), width), dtype=np.int64)
    for i, r in enumerate(rows):
        input_ids[i, : len(r)] = r
        mask[i, : len(r)] = 1
    return input_ids, mask


def batches(df, shuffle):
    idx = np.arange(len(df))
    return DataLoader(idx, batch_size=CFG["batch_size"], shuffle=shuffle)


def to_tensors(df, idx):
    ids, mask = encode(backend, df["text_clean"].iloc[idx.numpy()].tolist())
    return torch.from_numpy(ids).to(device), torch.from_numpy(mask).to(device)


@torch.no_grad()
def predict(model, df):
    model.eval()
    out = []
    for idx in batches(df, shuffle=False):
        ids, mask = to_tensors(df, idx)
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=device == "cuda"):
            logits = model(input_ids=ids, attention_mask=mask).logits
        out.append(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(out)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(CFG["model_name"], num_labels=2)
model.to(device)
train = parts["train"].reset_index(drop=True)
y_train = torch.tensor(train["label_ad"].to_numpy())
loader = batches(train, shuffle=True)
steps = len(loader) * CFG["epochs"]
opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
sched = get_linear_schedule_with_warmup(opt, int(CFG["warmup"] * steps), steps)
loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor([1.0, CFG["pos_weight"]], device=device))
scaler = torch.amp.GradScaler(enabled=device == "cuda")

history, best = [], {"val_pr_auc": -1.0}
t0 = time.time()
for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    total = 0.0
    for step, idx in enumerate(loader):
        ids, mask = to_tensors(train, idx)
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=device == "cuda"):
            logits = model(input_ids=ids, attention_mask=mask).logits
        loss = loss_fn(logits.float(), y_train[idx].to(device))
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        total += loss.item()
        if step % 200 == 0:
            print(f"epoch {epoch} step {step}/{len(loader)} loss {loss.item():.4f}")
    p_val = predict(model, parts["val"])
    val_pr_auc = float(average_precision_score(parts["val"]["label_ad"], p_val))
    history.append({"epoch": epoch, "train_loss": total / len(loader), "val_pr_auc": val_pr_auc})
    print(history[-1], f"{(time.time() - t0) / 60:.1f} min")
    if val_pr_auc > best["val_pr_auc"]:  # keep the epoch with the best validation PR-AUC
        best = {"epoch": epoch, "val_pr_auc": val_pr_auc}
        best_state = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
train_minutes = (time.time() - t0) / 60

In [ ]:
model.load_state_dict(best_state)
preds = {}
for k in ("val", "test"):
    p = predict(model, parts[k])
    preds[k] = p
    parts[k][["channel", "id"]].assign(prob_ad=p).to_parquet(
        OUT_DIR / f"preds_{k}.parquet", index=False
    )
    print(k, "PR-AUC", average_precision_score(parts[k]["label_ad"], p))

In [ ]:
# Export to ONNX (fp32), then dynamic int8 quantization for CPU serving (~280 MB).
import onnxruntime as ort
from onnxruntime.quantization import QuantType, quantize_dynamic

model.save_pretrained(TMP_DIR / "hf_best")
export_model = AutoModelForSequenceClassification.from_pretrained(
    TMP_DIR / "hf_best", attn_implementation="eager"
).eval()


class LogitsOnly(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, input_ids, attention_mask):
        return self.m(input_ids=input_ids, attention_mask=attention_mask).logits


dummy = [torch.from_numpy(a) for a in encode(backend, ["приклад тексту", "ще один довший"])]
export_kwargs = dict(
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={"input_ids": {0: "batch", 1: "seq"}, "attention_mask": {0: "batch", 1: "seq"}},
    opset_version=17,
)
try:
    torch.onnx.export(
        LogitsOnly(export_model), tuple(dummy), TMP_DIR / "fp32.onnx", dynamo=False, **export_kwargs
    )
except TypeError:  # older torch without the dynamo argument
    torch.onnx.export(
        LogitsOnly(export_model), tuple(dummy), TMP_DIR / "fp32.onnx", **export_kwargs
    )
quantize_dynamic(TMP_DIR / "fp32.onnx", OUT_DIR / "model.onnx", weight_type=QuantType.QInt8)
backend.save(str(OUT_DIR / "tokenizer.json"))
print("model.onnx MB:", round((OUT_DIR / "model.onnx").stat().st_size / 1e6))

In [ ]:
# Check int8 ONNX against the GPU model on a test sample with every test ad in it.
test = parts["test"].reset_index(drop=True)
pos = test.index[test["label_ad"] == 1].tolist()
neg = (
    test.index[test["label_ad"] == 0]
    .to_series()
    .sample(min(400, int((test["label_ad"] == 0).sum())), random_state=0)
    .tolist()
)
check = test.loc[pos + neg]
sess = ort.InferenceSession(str(OUT_DIR / "model.onnx"), providers=["CPUExecutionProvider"])
p_onnx = []
for i in range(0, len(check), 16):
    ids, mask = encode(backend, check["text_clean"].iloc[i : i + 16].tolist())
    (logits,) = sess.run(["logits"], {"input_ids": ids, "attention_mask": mask})
    e = np.exp(logits - logits.max(axis=1, keepdims=True))
    p_onnx.append(e[:, 1] / e.sum(axis=1))
p_onnx = np.concatenate(p_onnx)
p_gpu = preds["test"][check.index.to_numpy()]
onnx_check = {
    "n": len(check),
    "max_abs_diff": float(np.abs(p_onnx - p_gpu).max()),
    "mean_abs_diff": float(np.abs(p_onnx - p_gpu).mean()),
    "pr_auc_gpu": float(average_precision_score(check["label_ad"], p_gpu)),
    "pr_auc_onnx_int8": float(average_precision_score(check["label_ad"], p_onnx)),
}
print(onnx_check)

In [ ]:
meta = {
    "params": CFG,
    "history": history,
    "best_epoch": best["epoch"],
    "train_minutes": round(train_minutes, 1),
    "manifest": manifest,
    "onnx_check": onnx_check,
    "max_rows": MAX_ROWS,
    "versions": {
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "onnxruntime": ort.__version__,
        "gpu": torch.cuda.get_device_name(0) if device == "cuda" else "cpu",
    },
}
(OUT_DIR / "train_meta.json").write_text(json.dumps(meta, indent=2, default=str))
shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
print("download:", f"{OUT_DIR}.zip")